# 00 — Environment Sanity Check

Verifies the full software stack before any real training.
Every cell should run **without errors**. Red = fix before proceeding.

## 1. Python & core imports

In [1]:
import sys, platform
print(f'Python  : {sys.version}')
print(f'Platform: {platform.platform()}')

Python  : 3.10.17 (main, Apr  9 2025, 03:56:18) [MSC v.1943 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0


In [2]:
# Torch
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} — device count: {torch.cuda.device_count()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using   : {DEVICE}')

PyTorch : 2.5.1+cu121
CUDA    : True — device count: 1
Using   : cuda


In [4]:
# Numpy / matplotlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
print(f'NumPy      : {np.__version__}')
print(f'Matplotlib : {matplotlib.__version__}')

NumPy      : 2.2.6
Matplotlib : 3.10.8


In [5]:
# Optional but recommended
for pkg in ['minari', 'gymnasium', 'h5py', 'wandb', 'tensorboard']:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'installed')
        print(f'  ✓  {pkg:<15} {ver}')
    except ImportError:
        print(f'  ✗  {pkg:<15} NOT installed')

  ✓  minari          0.5.3
  ✓  gymnasium       1.2.3
  ✓  h5py            3.16.0
  ✓  wandb           0.25.1
  ✓  tensorboard     2.20.0


## 2. Path setup — make sure offline_sac modules are importable

In [7]:
import sys
from pathlib import Path

# Notebooks live in offline_sac/01_notebooks/
# Source lives one level up
SRC = Path('..').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print(f'Source root: {SRC}')
print('Files found:', [f.name for f in SRC.glob('*.py')])

Source root: D:\01 Work\12-Recommendations\01 Offline_SAC
Files found: ['buffer.py', 'dataset_loader.py', 'evaluator.py', 'networks.py', 'run_logger.py', 'smoke_test.py', 'train.py', 'trainer.py']


## 3. Import project classes

In [8]:
from buffer import OfflineReplayBuffer, Batch
from dataset_loader import DatasetLoader
from networks import Actor, TwinQCritic
from trainer import SACTrainer, SACConfig
print('All project imports OK')

All project imports OK


## 4. Buffer — synthetic round-trip

In [9]:
loader = DatasetLoader(device=DEVICE)
buf    = loader.load('hopper-medium-v2', backend='synthetic')
print(buf)

batch = buf.sample(256)
print(f'obs      : {batch.obs.shape}   dtype={batch.obs.dtype}   device={batch.obs.device}')
print(f'actions  : {batch.actions.shape}')
print(f'rewards  : {batch.rewards.shape}  min={batch.rewards.min():.2f} max={batch.rewards.max():.2f}')
print(f'dones    : {batch.dones.unique()}')

assert batch.dones.max() <= 1.0, 'dones should be binary'
print('Buffer sample OK')

Using SYNTHETIC dataset for 'hopper-medium-v2' — only for pipeline smoke tests!


OfflineReplayBuffer(size=200,000, obs_dim=11, act_dim=3, reward=[-4.37, 5.31], device=cuda)
obs      : torch.Size([256, 11])   dtype=torch.float32   device=cuda:0
actions  : torch.Size([256, 3])
rewards  : torch.Size([256, 1])  min=-2.57 max=2.91
dones    : tensor([0.], device='cuda:0')
Buffer sample OK


## 5. Networks — forward pass shapes

In [10]:
OBS_DIM, ACT_DIM = buf.obs_dim, buf.act_dim

actor  = Actor(OBS_DIM, ACT_DIM, hidden_dim=64).to(DEVICE)
critic = TwinQCritic(OBS_DIM, ACT_DIM, hidden_dim=64).to(DEVICE)

obs_t  = batch.obs[:8]
act_t  = batch.actions[:8]

with torch.no_grad():
    actions, log_pi = actor(obs_t)
    q1, q2          = critic(obs_t, act_t)

print(f'Actor  → actions {actions.shape}  log_pi {log_pi.shape}')
print(f'Critic → q1 {q1.shape}  q2 {q2.shape}')
assert actions.abs().max().item() < 1.0 + 1e-4, 'tanh bound violated'
assert torch.isfinite(log_pi).all(), 'log_pi has NaN/Inf'
print('Network forward pass OK')

Actor  → actions torch.Size([8, 3])  log_pi torch.Size([8])
Critic → q1 torch.Size([8, 1])  q2 torch.Size([8, 1])
Network forward pass OK


## 6. Trainer — 50 gradient steps

In [11]:
cfg     = SACConfig(hidden_dim=64, batch_size=64, log_interval=10)
trainer = SACTrainer(OBS_DIM, ACT_DIM, config=cfg, device=DEVICE)

losses = []
for i in range(50):
    m = trainer.update(buf)
    if m:
        losses.append(m)
        print(f"  step={m['step']:3d}  critic={m['critic_loss']:.4f}  ",
              f"actor={m['actor_loss']:.4f}  alpha={m['alpha']:.4f}")

assert all(torch.isfinite(torch.tensor(l['critic_loss'])) for l in losses), \
    'NaN in critic loss'
print('Trainer 50-step run OK')

  step= 10  critic=8.0391   actor=-0.5985  alpha=0.9970
  step= 20  critic=6.1734   actor=-0.6458  alpha=0.9939
  step= 30  critic=5.5408   actor=-1.4932  alpha=0.9908
  step= 40  critic=9.1048   actor=-1.4853  alpha=0.9876
  step= 50  critic=5.3303   actor=-1.7546  alpha=0.9844
Trainer 50-step run OK


## 7. Checkpoint round-trip

In [12]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'ckpt.pt')
    torch.save(trainer.state_dict(), path)

    trainer2 = SACTrainer(OBS_DIM, ACT_DIM, config=cfg, device=DEVICE)
    trainer2.load_state_dict(torch.load(path, weights_only=True))

    assert trainer2.step == trainer.step
    for p1, p2 in zip(trainer.actor.parameters(), trainer2.actor.parameters()):
        assert torch.allclose(p1, p2)

print('Checkpoint round-trip OK')

Checkpoint round-trip OK


## Summary

In [13]:
print()
print('='*50)
print('  All sanity checks passed.')
print(f'  Device: {DEVICE}')
print(f'  obs_dim={OBS_DIM}  act_dim={ACT_DIM}')
print('='*50)


  All sanity checks passed.
  Device: cuda
  obs_dim=11  act_dim=3
